# PixelStream-SR
## VAE-free, one-step, real-time face video super-resolution with a clean-pixel-prediction transformer

**Why this project.** Almost every video generation / editing stack today runs diffusion inside a VAE latent space. The VAE costs decode latency, blocks pixel-exact conditioning and pixel-space perceptual losses, and smears exactly the details that matter for talking-face video (teeth, eyelashes, skin texture). This notebook builds a **Diffusion Transformer that works directly on pixels** and shows that it can be *both* fast and sharp.

**The idea — three ingredients that only work well together:**

1. **Clean-pixel prediction (x-prediction) flow matching on the HR − bicubic residual, with 8×8 pixel patches.** Predicting the *clean image* keeps the network on the low-dimensional image manifold even though the noise it receives is high-dimensional. Diffusing the residual means the model only has to transport the *detail layer*, so **a single Euler step already gives a sharp result**.
2. **Representation alignment in pixel space.** A mid-layer of the DiT is aligned to frozen **DINOv2** features of the clean frame (REPA-style). In latent diffusion the VAE latent already carries semantics; in pixel space there is none, so this is where the semantic prior comes from. Zero cost at inference.
3. **Shared-noise streaming.** Reusing one noise seed across frames removes most diffusion flicker at zero cost, with no state and no drift. We also train and study **recurrence on the model's own previous output** (cells 9 and 13): it helps only under fast motion and drifts over long clips, so it is reported as an ablation, not as the method.

**What you will see (all in this notebook):**
* ×4 face super-resolution (32 → 128 px) that beats bicubic by a large margin in PSNR / SSIM / LPIPS.
* A table of quality vs. number of steps vs. **latency in milliseconds**.
* A side-by-side video: LR | bicubic | per-frame diffusion (flickers) | **ours, shared noise** (stable) | ground truth, with a temporal-consistency score and a drift check.
* Optional: upload your own face video and super-resolve it.


In [ ]:
#@title 1 · Setup — installs, imports, GPU check (run once)
import subprocess, sys
def pip(*pkgs): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
pip("datasets>=3.0.0", "lpips", "imageio", "imageio-ffmpeg", "scikit-image", "timm")

import os, math, time, copy, random, json
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

random.seed(0); np.random.seed(0); torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)
if device.type == "cuda":
    _p = torch.cuda.get_device_properties(0)
    print(f"GPU: {_p.name} ({_p.total_memory / 2**30:.0f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
BF16 = device.type == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8   # Ampere+ (A100/L4/A10); T4 uses fp16
AMP_DTYPE = torch.bfloat16 if BF16 else torch.float16
USE_AMP = device.type == "cuda"
print("mixed precision:", AMP_DTYPE if USE_AMP else "off")

In [ ]:
#@title 2 · Config — auto-tuned to your GPU (edit freely)
MOUNT_DRIVE = True  #@param {type:"boolean"}
RUN_NAME = "pixelstream_final"  #@param {type:"string"}   # use "pixelstream_norepa" with USE_REPA=False for the ablation
SAVE_DIR = f"/content/{RUN_NAME}"
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        SAVE_DIR = f"/content/drive/MyDrive/Aug26/VAE_free/{RUN_NAME}"        # survives a runtime reset
    except Exception as e:
        print("Drive not mounted (" + repr(e)[:60] + "); saving locally instead")
os.makedirs(SAVE_DIR, exist_ok=True)

IMG, SCALE = 128, 4                 # HR 128 px, LR 32 px (×4 super-resolution)
LR_SIZE = IMG // SCALE
PATCH, DIM, DEPTH, HEADS = 8, 384, 10, 6     # 16×16 = 256 pixel tokens, ~30M params
USE_REPA = True  #@param {type:"boolean"}
REPA_LAYER, LAMBDA_REPA = 4, 0.5    # align the 4th block's tokens to DINOv2-S patch tokens
RES_SCALE = 3.0                     # the residual (HR − bicubic) is multiplied by this before diffusion
PREV_KEEP_P = 0.6                   # how often the previous frame is given during training (rest: zeros = first frame)
PREV_MOTION = dict(max_shift=0.02, max_rot=1.0, max_scale=0.02)   # frame-to-frame motion simulated in training (~1–2 px, like real video)
DEGRADE = True                      # random blur + noise on the LR input (blind-ish SR; helps on real videos)

gpu_name = torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu"
if any(k in gpu_name for k in ["A100", "H100", "H200", "B200"]):
    N_TRAIN, BATCH, STEPS, LR = 30000, 96, 14000, 3e-4
elif any(k in gpu_name for k in ["L4", "A10", "L40", "RTX 6000", "V100"]):
    N_TRAIN, BATCH, STEPS, LR = 24000, 64, 12000, 3e-4
else:                               # T4 or unknown
    N_TRAIN, BATCH, STEPS, LR = 16000, 48, 10000, 2.5e-4
N_VAL = 256
print(f"{gpu_name}: train images={N_TRAIN}, batch={BATCH}, steps={STEPS}, lr={LR}")
print("checkpoints ->", SAVE_DIR)

In [ ]:
#@title 3 · Data — CelebA faces at 128 px, streamed from the Hugging Face Hub and cached in RAM
from datasets import load_dataset

def _first_image(ex):
    for v in ex.values():
        if isinstance(v, Image.Image): return v
    raise KeyError(f"no image column in {list(ex.keys())}")

t0 = time.time()
ds = load_dataset("tglcourse/CelebA-faces-cropped-128", split="train", streaming=True)
N_ALL = N_TRAIN + N_VAL
imgs = np.empty((N_ALL, IMG, IMG, 3), dtype=np.uint8)
for i, ex in enumerate(ds.take(N_ALL)):
    im = _first_image(ex).convert("RGB")
    if im.size != (IMG, IMG): im = im.resize((IMG, IMG), Image.BICUBIC)
    imgs[i] = np.asarray(im)
    if i % 5000 == 0: print(f"  {i}/{N_ALL}  ({time.time()-t0:.0f}s)")
data_u8 = torch.from_numpy(imgs)                      # (N, H, W, 3) uint8, NHWC, no copy
train_u8, val_u8 = data_u8[:N_TRAIN], data_u8[N_TRAIN:]
print("train", tuple(train_u8.shape), "val", tuple(val_u8.shape), f"({time.time()-t0:.0f}s)")

def to_float(u8):   # NHWC uint8 (cpu) -> NCHW float in [-1, 1] on device
    return u8.to(device, non_blocking=True).permute(0, 3, 1, 2).float().div_(127.5).sub_(1.0)

plt.figure(figsize=(12, 3)); plt.imshow(np.concatenate(list(imgs[:8]), 1)); plt.axis("off"); plt.title("training faces"); plt.show()

In [ ]:
#@title 4 · Degradation model, motion warps and helpers
def gaussian_blur(x, sigma, k=7):
    ax = torch.arange(k, device=x.device, dtype=torch.float32) - (k - 1) / 2
    g = torch.exp(-ax**2 / (2 * sigma**2)); g = g / g.sum()
    kernel = (g[:, None] * g[None, :]).expand(x.shape[1], 1, k, k).contiguous()
    return F.conv2d(F.pad(x, (k//2,)*4, mode="reflect"), kernel, groups=x.shape[1])

def make_lr(hr, degrade=True):
    '''HR (B,3,128,128) in [-1,1] -> LR (B,3,32,32). Random blur/noise = a mild blind-SR degradation.'''
    x = hr
    if degrade and random.random() < 0.6: x = gaussian_blur(x, sigma=random.uniform(0.3, 1.6))
    lr = F.interpolate(x, size=(LR_SIZE, LR_SIZE), mode="area")
    if degrade and random.random() < 0.5: lr = lr + torch.randn_like(lr) * random.uniform(0.0, 0.04)
    return lr.clamp(-1, 1)

def upsample(lr):
    return F.interpolate(lr, size=(IMG, IMG), mode="bicubic", align_corners=False).clamp(-1, 1)

def affine_theta(tx, ty, rot_deg, scale):
    '''(B,) tensors -> theta (B,2,3) for F.affine_grid (maps output coords to input coords).'''
    r = rot_deg * math.pi / 180
    c, s = torch.cos(r) * scale, torch.sin(r) * scale
    return torch.stack([torch.stack([c, -s, tx], -1), torch.stack([s, c, ty], -1)], 1)

def warp(x, theta):
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x.contiguous(), grid, mode="bilinear", padding_mode="reflection", align_corners=False)

def random_theta(B, max_shift=0.06, max_rot=3.0, max_scale=0.05):
    u = lambda: torch.rand(B, device=device) * 2 - 1
    return affine_theta(u() * max_shift, u() * max_shift, u() * max_rot, 1 + u() * max_scale)

def invert_theta(theta):
    B = theta.shape[0]
    row = torch.tensor([0., 0., 1.], device=theta.device).expand(B, 1, 3)
    return torch.inverse(torch.cat([theta, row], 1))[:, :2]

def make_prev_condition(hr):
    '''Simulated 'previous super-resolved frame': a slightly moved, slightly imperfect copy of the HR image.
    Dropped (zeros, mask=0) part of the time so the model also works on the first frame of a stream.'''
    B = hr.shape[0]
    prev = warp(hr, random_theta(B, **PREV_MOTION))
    prev = prev + torch.randn_like(prev) * random.uniform(0.0, 0.06)
    if random.random() < 0.3: prev = gaussian_blur(prev, random.uniform(0.3, 1.0))
    keep = (torch.rand(B, 1, 1, 1, device=device) < PREV_KEEP_P).float()
    return prev.clamp(-1, 1) * keep, keep.expand(B, 1, IMG, IMG).contiguous()

def build_cond(lr_up, prev, mask):
    return torch.cat([lr_up, prev, mask], 1)        # (B, 7, H, W)

@torch.no_grad()
def psnr(a, b):
    mse = ((a.clamp(-1, 1) - b.clamp(-1, 1)) / 2).pow(2).flatten(1).mean(1)
    return 10 * torch.log10(1.0 / mse.clamp_min(1e-10))

def to_img(x):   # (B,3,H,W) in [-1,1] -> (B,H,W,3) uint8
    return ((x.clamp(-1, 1) + 1) * 127.5).round().byte().permute(0, 2, 3, 1).cpu().numpy()

# sanity check of the warp / inverse-warp pair
_x = to_float(val_u8[:2]); _th = random_theta(2)
_err = (warp(warp(_x, _th), invert_theta(_th)) - _x)[:, :, IMG//4:-IMG//4, IMG//4:-IMG//4].abs().mean().item()
print(f"warp→inverse-warp round-trip error on real faces (interpolation only, should be ~0.01–0.03): {_err:.4f}")

In [ ]:
#@title 5 · Model — PixelDiT: a pixel-space Diffusion Transformer with adaLN-Zero, RMSNorm, QK-norm and a REPA head
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).to(x.dtype)

def modulate(x, shift, scale):
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.h, self.dh = heads, dim // heads
        self.qkv = nn.Linear(dim, 3 * dim); self.proj = nn.Linear(dim, dim)
        self.qn, self.kn = RMSNorm(self.dh), RMSNorm(self.dh)        # QK-norm: stable in bf16
    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        q, k = self.qn(q), self.kn(k)
        x = F.scaled_dot_product_attention(q, k, v)
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio=4.0):
        super().__init__()
        self.n1, self.attn = RMSNorm(dim), Attention(dim, heads)
        self.n2 = RMSNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(approximate="tanh"), nn.Linear(int(dim * mlp_ratio), dim))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.ada[1].weight); nn.init.zeros_(self.ada[1].bias)   # adaLN-Zero: blocks start as identity
    def forward(self, x, c):
        s1, sc1, g1, s2, sc2, g2 = self.ada(c).chunk(6, dim=-1)
        x = x + g1.unsqueeze(1) * self.attn(modulate(self.n1(x), s1, sc1))
        x = x + g2.unsqueeze(1) * self.mlp(modulate(self.n2(x), s2, sc2))
        return x

def timestep_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, device=t.device, dtype=torch.float32) / half)
    args = t.float()[:, None] * 1000.0 * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], -1)

def sincos_2d(dim, grid):
    ys, xs = torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij")
    def emb(pos):
        d = dim // 2
        omega = 1.0 / (10000 ** (torch.arange(d // 2, dtype=torch.float32) / (d // 2)))
        out = pos.float()[:, None] * omega[None]
        return torch.cat([torch.sin(out), torch.cos(out)], -1)
    return torch.cat([emb(ys.flatten()), emb(xs.flatten())], -1)        # (grid*grid, dim)

class PixelDiT(nn.Module):
    '''Input: noisy residual x_t (3ch) ++ conditioning (7ch: bicubic LR, previous frame, validity mask).
    Output: the predicted CLEAN residual x0 (x-prediction). Also returns mid-layer tokens for REPA.'''
    def __init__(self, img=128, patch=8, in_ch=10, out_ch=3, dim=384, depth=10, heads=6, repa_layer=4, repa_dim=384):
        super().__init__()
        self.p, self.out_ch, self.grid = patch, out_ch, img // patch
        self.x_embed = nn.Linear(in_ch * patch * patch, dim)                        # linear patchify
        self.register_buffer("pos", sincos_2d(dim, self.grid), persistent=False)
        self.t_embed = nn.Sequential(nn.Linear(256, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.blocks = nn.ModuleList([Block(dim, heads) for _ in range(depth)])
        self.repa_layer = repa_layer
        self.repa_head = nn.Sequential(nn.Linear(dim, 1024), nn.SiLU(), nn.Linear(1024, repa_dim))
        self.final_norm = RMSNorm(dim)
        self.final_ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 2 * dim))
        self.final = nn.Linear(dim, out_ch * patch * patch)
        for m in (self.final, self.final_ada[1]): nn.init.zeros_(m.weight); nn.init.zeros_(m.bias)

    def patchify(self, x):
        B, C, H, W = x.shape; p, g = self.p, H // self.p
        return x.view(B, C, g, p, g, p).permute(0, 2, 4, 1, 3, 5).reshape(B, g * g, C * p * p)
    def unpatchify(self, x):
        B = x.shape[0]; p, g, C = self.p, self.grid, self.out_ch
        return x.view(B, g, g, C, p, p).permute(0, 3, 1, 4, 2, 5).reshape(B, C, g * p, g * p)

    def forward(self, x_t, t, cond, return_repa=False):
        h = self.x_embed(self.patchify(torch.cat([x_t, cond], 1))) + self.pos
        c = self.t_embed(timestep_embedding(t, 256))
        z = None
        for i, blk in enumerate(self.blocks):
            h = blk(h, c)
            if i == self.repa_layer - 1: z = h
        shift, scale = self.final_ada(c).chunk(2, -1)
        out = self.unpatchify(self.final(modulate(self.final_norm(h), shift, scale)))
        return (out, self.repa_head(z)) if return_repa else out

_m = PixelDiT(img=IMG, patch=PATCH, in_ch=10, dim=DIM, depth=DEPTH, heads=HEADS, repa_layer=REPA_LAYER)
print(f"PixelDiT: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M params, {(IMG//PATCH)**2} tokens of {PATCH}×{PATCH} pixels")
del _m

In [ ]:
#@title 6 · Frozen DINOv2-S teacher for representation alignment (REPA, but in pixel space)
dino, _dino_fwd = None, None
if USE_REPA:
    try:
        dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", trust_repo=True, verbose=False)
        _dino_fwd = lambda x: dino.forward_features(x)["x_norm_patchtokens"]
    except Exception as e:
        print("torch.hub failed:", repr(e)[:160], "-> trying timm")
        try:
            import timm
            dino = timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True, img_size=224, num_classes=0)
            _dino_fwd = lambda x: dino.forward_features(x)[:, 1:]      # drop CLS
        except Exception as e2:
            print("timm failed too:", repr(e2)[:160], "-> REPA disabled for this run"); USE_REPA = False

if USE_REPA:
    dino = dino.to(device).eval().requires_grad_(False)
    IM_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    IM_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.no_grad()
    def dino_tokens(hr):   # hr in [-1,1] at 128 px -> (B, 256, 384) patch tokens on the same 16×16 grid as the DiT
        x = F.interpolate((hr + 1) / 2, size=(224, 224), mode="bicubic", align_corners=False).clamp(0, 1)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            f = _dino_fwd((x - IM_MEAN) / IM_STD)
        return f.float()

    _f = dino_tokens(to_float(val_u8[:2]))
    assert _f.shape[1] == (IMG // PATCH) ** 2, f"token grid mismatch {_f.shape}"
    print("DINOv2 tokens:", tuple(_f.shape), "(matches the DiT token grid)")
else:
    print("REPA off")

In [ ]:
#@title 7 · Train — loss curve + live previews. Re-run this cell to continue training (it resumes from the checkpoint).
model = PixelDiT(img=IMG, patch=PATCH, in_ch=10, out_ch=3, dim=DIM, depth=DEPTH, heads=HEADS, repa_layer=REPA_LAYER).to(device)
ema = copy.deepcopy(model).eval().requires_grad_(False)
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.02)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and AMP_DTYPE == torch.float16))
WARMUP = 300
def lr_at(step):
    if step < WARMUP: return LR * (step + 1) / WARMUP
    p = (step - WARMUP) / max(1, STEPS - WARMUP)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))

ckpt_path = f"{SAVE_DIR}/pixelstream.pt"
step0, hist = 0, {"step": [], "loss": [], "loss_x": [], "loss_repa": []}
if os.path.exists(ckpt_path):
    ck = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ck["model"]); ema.load_state_dict(ck["ema"]); opt.load_state_dict(ck["opt"])
    step0, hist = ck["step"], ck["hist"]; print(f"resumed from step {step0}")

def sample_t(B):                       # logit-normal timestep sampling (SD3 / JiT style)
    return torch.sigmoid(torch.randn(B, device=device)).clamp(1e-3, 1.0)

@torch.no_grad()
def sample(net, cond, steps=4, noise=None):
    '''Deterministic Euler sampler for the x-prediction flow: x_{t'} = x̂0 + (t'/t)(x_t − x̂0).
    steps=1 is a single forward pass (direct clean-pixel prediction).'''
    B = cond.shape[0]
    x = torch.randn(B, 3, IMG, IMG, device=device) if noise is None else noise
    ts = torch.linspace(1.0, 0.0, steps + 1, device=device)
    for i in range(steps):
        t, t_next = ts[i], ts[i + 1]
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            x0_hat = net(x, t.expand(B), cond).float()
        x = x0_hat + (t_next / t) * (x - x0_hat)
    return (cond[:, :3] + x / RES_SCALE).clamp(-1, 1)

fixed_hr = to_float(val_u8[:4]); fixed_up = upsample(make_lr(fixed_hr, degrade=False))
fixed_cond = build_cond(fixed_up, torch.zeros_like(fixed_hr), torch.zeros(4, 1, IMG, IMG, device=device))
def preview(step):
    g = torch.Generator(device=device).manual_seed(0)
    noise = torch.randn(4, 3, IMG, IMG, device=device, generator=g)
    o1, o4 = sample(ema, fixed_cond, 1, noise.clone()), sample(ema, fixed_cond, 4, noise.clone())
    rows = [np.concatenate(list(to_img(r)), 1) for r in (fixed_up, o1, o4, fixed_hr)]
    plt.figure(figsize=(10, 10)); plt.imshow(np.concatenate(rows, 0)); plt.axis("off")
    plt.title(f"step {step} — rows: bicubic ×4 | ours 1-step | ours 4-step | ground truth")
    plt.savefig(f"{SAVE_DIR}/preview_{step}.png", bbox_inches="tight", dpi=80); plt.show()

def save_ckpt(step):
    torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "opt": opt.state_dict(), "step": step, "hist": hist,
                "cfg": dict(IMG=IMG, SCALE=SCALE, PATCH=PATCH, DIM=DIM, DEPTH=DEPTH, HEADS=HEADS, RES_SCALE=RES_SCALE, USE_REPA=USE_REPA)}, ckpt_path)

model.train(); t_start = time.time(); run = {"loss": 0., "loss_x": 0., "loss_repa": 0., "n": 0}
for step in range(step0, STEPS):
    for g in opt.param_groups: g["lr"] = lr_at(step)
    hr = to_float(train_u8[torch.randint(0, N_TRAIN, (BATCH,))])
    if random.random() < 0.5: hr = hr.flip(3)
    lr_up = upsample(make_lr(hr, DEGRADE))
    prev, mask = make_prev_condition(hr)
    cond = build_cond(lr_up, prev, mask)
    x0 = (hr - lr_up) * RES_SCALE                       # clean target: the detail layer
    t = sample_t(BATCH); tb = t.view(-1, 1, 1, 1)
    x_t = (1 - tb) * x0 + tb * torch.randn_like(x0)     # linear flow-matching path (t=0 clean, t=1 noise)
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        pred, z = model(x_t, t, cond, return_repa=True)
    loss_x = F.mse_loss(pred.float(), x0)
    loss_repa = torch.zeros((), device=device)
    if USE_REPA:
        loss_repa = (1 - F.cosine_similarity(z.float(), dino_tokens(hr), dim=-1)).mean()
    loss = loss_x + LAMBDA_REPA * loss_repa
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    decay = min(0.9995, (1 + step) / (10 + step))
    with torch.no_grad():
        for pe, pm in zip(ema.parameters(), model.parameters()): pe.lerp_(pm, 1 - decay)

    for k, v in (("loss", loss), ("loss_x", loss_x), ("loss_repa", loss_repa)): run[k] += v.item()
    run["n"] += 1
    if (step + 1) % 50 == 0:
        hist["step"].append(step + 1)
        for k in ("loss", "loss_x", "loss_repa"): hist[k].append(run[k] / run["n"]); run[k] = 0.
        run["n"] = 0
    if (step + 1) % 100 == 0:
        el = time.time() - t_start; rate = (step + 1 - step0) / el
        print(f"step {step+1:6d}/{STEPS} | loss {hist['loss'][-1]:.4f} (x {hist['loss_x'][-1]:.4f}, repa {hist['loss_repa'][-1]:.3f}) "
              f"| lr {lr_at(step):.1e} | {rate:.2f} it/s | ETA {(STEPS-step-1)/rate/60:.0f} min")
    if (step + 1) % 1000 == 0 or step + 1 == STEPS:
        save_ckpt(step + 1); preview(step + 1)

if hist["step"]:
    plt.figure(figsize=(8, 3)); plt.plot(hist["step"], hist["loss_x"], label="pixel loss"); plt.plot(hist["step"], hist["loss_repa"], label="REPA loss")
    plt.yscale("log"); plt.xlabel("step"); plt.legend(); plt.title("training curves"); plt.show()
print(f"done. checkpoint: {ckpt_path}")

In [ ]:
#@title 8 · Evaluate — PSNR / SSIM / LPIPS vs bicubic, and latency per number of steps
import lpips
from skimage.metrics import structural_similarity as ssim_fn
lp = lpips.LPIPS(net="alex", verbose=False).to(device).eval()

val = to_float(val_u8[:N_VAL])
val_up = upsample(F.interpolate(val, size=(LR_SIZE, LR_SIZE), mode="area"))
val_cond = build_cond(val_up, torch.zeros_like(val), torch.zeros(N_VAL, 1, IMG, IMG, device=device))

def metrics(out, ref):
    p = psnr(out, ref).mean().item()
    s = float(np.mean([ssim_fn(a, b, channel_axis=2, data_range=255) for a, b in zip(to_img(out), to_img(ref))]))
    with torch.no_grad(): l = float(np.mean([lp(out[i:i+64], ref[i:i+64]).mean().item() for i in range(0, len(out), 64)]))
    return p, s, l

def run_batched(cond, steps, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    outs = []
    for i in range(0, cond.shape[0], 64):
        c = cond[i:i+64]; outs.append(sample(ema, c, steps, torch.randn(c.shape[0], 3, IMG, IMG, device=device, generator=g)))
    return torch.cat(outs)

def latency_ms(steps, n=50):
    c = val_cond[:1]
    for _ in range(5): sample(ema, c, steps)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): sample(ema, c, steps)
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

results = {"bicubic ×4": (*metrics(val_up, val), float("nan"))}
outs_by_steps = {}
for k in (1, 2, 4, 8):
    outs_by_steps[k] = run_batched(val_cond, k)
    results[f"ours {k}-step"] = (*metrics(outs_by_steps[k], val), latency_ms(k))

print(f"{'method':<14}{'PSNR↑':>8}{'SSIM↑':>8}{'LPIPS↓':>8}{'ms/frame (bs=1)':>18}")
for name, (p, s, l, ms) in results.items(): print(f"{name:<14}{p:8.2f}{s:8.3f}{l:8.3f}{ms:18.1f}")
json.dump(results, open(f"{SAVE_DIR}/results.json", "w"), indent=1)

# qualitative grid: LR (nearest) | bicubic | 1-step | 4-step | ground truth
idx = list(range(6))
lr_show = F.interpolate(F.interpolate(val[idx], size=(LR_SIZE, LR_SIZE), mode="area"), scale_factor=SCALE, mode="nearest")
cols = [lr_show, val_up[idx], outs_by_steps[1][idx], outs_by_steps[4][idx], val[idx]]
grid = np.concatenate([np.concatenate(list(to_img(c)), 0) for c in cols], 1)
plt.figure(figsize=(12, 14)); plt.imshow(grid); plt.axis("off")
plt.title("LR 32px | bicubic | ours 1-step | ours 4-step | ground truth"); plt.show()
Image.fromarray(grid).save(f"{SAVE_DIR}/qualitative.png")

In [ ]:
#@title 9 · Streaming video demo — synthetic camera motion: per-frame vs shared noise (+ recurrence ablation)
import imageio.v2 as iio
from IPython.display import HTML, display
from base64 import b64encode

T_FRAMES = 32
def motion_thetas(T, seed=0, shift=0.06, rot=3.0, zoom=0.05):
    g = torch.Generator().manual_seed(seed)
    ph = torch.rand(4, generator=g) * 2 * math.pi; fr = 1 + torch.rand(4, generator=g)
    k = torch.arange(T).float()
    tx = shift * torch.sin(2 * math.pi * fr[0] * k / T + ph[0]); ty = 0.7 * shift * torch.sin(2 * math.pi * fr[1] * k / T + ph[1])
    rd = rot * torch.sin(2 * math.pi * fr[2] * k / T + ph[2]); sc = 1 + zoom * torch.sin(2 * math.pi * fr[3] * k / T + ph[3])
    return affine_theta(tx, ty, rd, sc).to(device)

def make_sequence(hr_img, thetas):        # (1,3,H,W) -> (T,3,H,W) ground-truth frames with smooth camera motion
    return warp(hr_img.expand(thetas.shape[0], -1, -1, -1), thetas)

@torch.no_grad()
def stream_sr(lr_frames, steps=4, recurrent=True, shared_noise=True, seed=0):
    '''Online, frame-by-frame super-resolution. recurrent: condition on our own previous output.
    shared_noise: reuse one noise tensor for all frames (free temporal regulariser).'''
    g = torch.Generator(device=device).manual_seed(seed)
    base_noise = torch.randn(1, 3, IMG, IMG, device=device, generator=g)
    prev, mask = torch.zeros(1, 3, IMG, IMG, device=device), torch.zeros(1, 1, IMG, IMG, device=device)
    outs = []
    for lr in lr_frames:
        noise = base_noise.clone() if shared_noise else torch.randn(1, 3, IMG, IMG, device=device, generator=g)
        out = sample(ema, build_cond(upsample(lr[None]), prev, mask), steps, noise)
        outs.append(out)
        if recurrent: prev, mask = out, torch.ones_like(mask)
    return torch.cat(outs)

def flicker(frames, thetas):
    '''Temporal inconsistency: un-warp every frame back to the reference pose, then average |frame_t − frame_{t−1}|
    over the central crop (8-bit units, lower is better). Smooth outputs (bicubic) score low by construction;
    the ground-truth row is the floor for *sharp* content, so compare 'ours' against it.'''
    un = warp(frames, invert_theta(thetas)); c = IMG // 4
    return (un[1:, :, c:-c, c:-c] - un[:-1, :, c:-c, c:-c]).abs().mean().item() * 127.5

def write_video(path, panels, labels, fps=12, up=2):
    frames = []
    for i in range(panels[0].shape[0]):
        tiles = []
        for p, lab in zip(panels, labels):
            im = Image.fromarray(to_img(p[i:i+1])[0]).resize((IMG * up, IMG * up), Image.NEAREST)
            d = ImageDraw.Draw(im); d.rectangle([0, 0, IMG * up, 14], fill=(0, 0, 0)); d.text((4, 1), lab, fill=(255, 255, 255))
            tiles.append(np.asarray(im))
        frames.append(np.concatenate(tiles, 1))
    iio.mimwrite(path, frames, fps=fps, codec="libx264", quality=8, pixelformat="yuv420p", macro_block_size=1)
    iio.mimwrite(path.replace(".mp4", ".gif"), frames[::2], fps=max(1, fps // 2))
    return path

def show_video(path, width=1000):
    display(HTML(f'<video width="{width}" controls autoplay loop muted src="data:video/mp4;base64,{b64encode(open(path, "rb").read()).decode()}"></video>'))

VARIANTS = {"per-frame (indep. noise)":      dict(recurrent=False, shared_noise=False),
            "ours: shared noise":            dict(recurrent=False, shared_noise=True),
            "recurrent + shared (ablation)": dict(recurrent=True,  shared_noise=True)}
VIDEO_FACES, VIDEO_STEPS = (8, 21, 33), (1, 4)        # 1 step = real-time mode, 4 steps = quality mode
video_results, videos = {}, {}
for steps_v in VIDEO_STEPS:
    rows = {k: {"flicker": [], "psnr": []} for k in list(VARIANTS) + ["bicubic", "ground truth"]}
    for vi, face in enumerate(VIDEO_FACES):
        thetas = motion_thetas(T_FRAMES, seed=vi)
        gt = make_sequence(to_float(val_u8[face:face+1]), thetas)
        lr_frames = F.interpolate(gt, size=(LR_SIZE, LR_SIZE), mode="area")
        bic = upsample(lr_frames)
        outs = {name: stream_sr(lr_frames, steps_v, **kw) for name, kw in VARIANTS.items()}
        for name, o in list(outs.items()) + [("bicubic", bic), ("ground truth", gt)]:
            rows[name]["flicker"].append(flicker(o, thetas)); rows[name]["psnr"].append(psnr(o, gt).mean().item())
        if vi == 0:
            panels = [F.interpolate(lr_frames, scale_factor=SCALE, mode="nearest"), bic, outs["per-frame (indep. noise)"], outs["ours: shared noise"], gt]
            videos[steps_v] = write_video(f"{SAVE_DIR}/demo_{steps_v}step.mp4", panels, ["LR 32px", "bicubic", "per-frame diffusion", "OURS (shared noise)", "ground truth"])
            drift_psnr = {name: psnr(outs[name], gt) for name in ("ours: shared noise", "recurrent + shared (ablation)")}
    print(f"\n{'variant':<26}{'flicker↓':>10}{'PSNR↑':>8}   ({steps_v} step(s)/frame, mean over {len(VIDEO_FACES)} clips)")
    for name, r in rows.items(): print(f"{name:<26}{np.mean(r['flicker']):10.2f}{np.mean(r['psnr']):8.2f}")
    for name, pf in drift_psnr.items():
        d = pf[-8:].mean().item() - pf[:8].mean().item()
        print(f"drift check ({name}, clip 0): PSNR first 8 frames {pf[:8].mean():.2f} → last 8 frames {pf[-8:].mean():.2f} "
              f"({'no drift' if abs(d) < 0.5 else 'DRIFT'})")
    video_results[f"{steps_v}-step"] = {k: {m: float(np.mean(v)) for m, v in r.items()} for k, r in rows.items()}
json.dump(video_results, open(f"{SAVE_DIR}/video_results.json", "w"), indent=1)
print("\nreal-time mode (1 step):"); show_video(videos[1])
print("quality mode (4 steps):"); show_video(videos[4])

In [ ]:
#@title 10 · Helpers for real videos (face crop → 32 px → streaming SR)
import cv2, urllib.request
_detector = None
if hasattr(cv2, "CascadeClassifier") and hasattr(cv2, "data"):            # OpenCV 4.x
    _haar = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    _detector = lambda rgb: [tuple(f) for f in _haar.detectMultiScale(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), 1.1, 5)]
    print("face detector: Haar cascade")
else:                                                                     # OpenCV 5 dropped Haar; use YuNet (DNN)
    try:
        _onnx = f"{SAVE_DIR}/yunet.onnx"
        if not os.path.exists(_onnx) or os.path.getsize(_onnx) < 100_000:
            urllib.request.urlretrieve("https://raw.githubusercontent.com/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx", _onnx)
            assert os.path.getsize(_onnx) > 100_000, "download did not return the model"
        _yunet = cv2.FaceDetectorYN.create(_onnx, "", (320, 320), 0.6, 0.3, 5000)
        def _detector(rgb):
            H, W = rgb.shape[:2]; _yunet.setInputSize((W, H))
            _, faces = _yunet.detect(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
            return [] if faces is None else [tuple(int(v) for v in f[:4]) for f in faces]
        print("face detector: YuNet")
    except Exception as e:
        print("no face detector available (" + repr(e)[:80] + ") -> using a centre crop; set FACE_BOX manually if needed")
FACE_BOX = None      # optional manual override: (centre_x, centre_y, size) in pixels of the uploaded video

def face_box(frame_rgb, margin=1.9):
    if FACE_BOX is not None: return FACE_BOX
    H, W = frame_rgb.shape[:2]
    faces = _detector(frame_rgb) if _detector is not None else []
    if len(faces) == 0: return W / 2, H / 2, int(min(H, W) * 0.8)
    x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
    return x + w / 2, y + h / 2, int(margin * max(w, h))

def square_crop(f, cx, cy, s):
    H, W = f.shape[:2]; x0, y0 = int(cx - s // 2), int(cy - s // 2)
    pad = max(0, -x0, -y0, x0 + s - W, y0 + s - H)
    if pad: f = cv2.copyMakeBorder(f, pad, pad, pad, pad, cv2.BORDER_REFLECT); x0 += pad; y0 += pad
    return f[y0:y0 + s, x0:x0 + s]

def frames_to_tensors(frames, max_frames=96):
    '''list of RGB uint8 frames -> (reference 128px, LR 32px) tensors. The box is fixed from the first frame.'''
    frames = frames[:max_frames]
    cx, cy, s = face_box(frames[0])
    ref = np.stack([cv2.resize(square_crop(f, cx, cy, s), (IMG, IMG), interpolation=cv2.INTER_AREA) for f in frames])
    ref = torch.from_numpy(ref).to(device).permute(0, 3, 1, 2).float().div(127.5).sub(1)
    return ref, F.interpolate(ref, size=(LR_SIZE, LR_SIZE), mode="area")

def super_resolve_video(frames, fps=12, steps=1, name="demo_upload"):
    """Final method: 1 step per frame + shared noise, no recurrence (recurrence drifts on long clips, see cell 13)."""
    ref, lr_frames = frames_to_tensors(frames)
    ours = stream_sr(lr_frames, steps, recurrent=False, shared_noise=True)
    base = stream_sr(lr_frames, steps, recurrent=False, shared_noise=False)
    print(f"{len(frames)} frames | PSNR vs reference: bicubic {psnr(upsample(lr_frames), ref).mean():.2f} | per-frame {psnr(base, ref).mean():.2f} | ours {psnr(ours, ref).mean():.2f}")
    panels = [F.interpolate(lr_frames, scale_factor=SCALE, mode="nearest"), upsample(lr_frames), base, ours, ref]
    return write_video(f"{SAVE_DIR}/{name}.mp4", panels, ["LR 32px", "bicubic", "per-frame diffusion", f"OURS ({steps}-step, shared noise)", "reference 128px"], fps=fps)
print("helpers ready")

In [ ]:
#@title 11 · (Optional) Upload your own face video — it is downscaled to 32 px and super-resolved live
from google.colab import files
up = files.upload()                      # pick a short .mp4 / .mov of a face (phone selfie video is perfect)
path = next(iter(up))
reader = iio.get_reader(path)
fps = int(round(reader.get_meta_data().get("fps", 25)))
frames = [f[..., :3] for i, f in enumerate(reader) if i < 96]
show_video(super_resolve_video(frames, fps=min(fps, 30)))

In [ ]:
#@title 12 · One-page summary of everything (also saved as summary.md)
lines = ["# PixelStream-SR — results", f"GPU: {gpu_name} | train steps: {STEPS} | batch: {BATCH} | params: {sum(p.numel() for p in ema.parameters())/1e6:.1f}M | REPA: {USE_REPA}", "",
         "## Single-image ×4 SR (32→128 px, 256 held-out CelebA faces)", "| method | PSNR↑ | SSIM↑ | LPIPS↓ | ms/frame |", "|---|---|---|---|---|"]
for name, (p, sv, l, ms) in results.items(): lines.append(f"| {name} | {p:.2f} | {sv:.3f} | {l:.3f} | {'' if ms != ms else f'{ms:.1f}'} |")
for k, rows_k in video_results.items():
    lines += ["", f"## Streaming video, {k} per frame (synthetic motion, 3 clips)", "| variant | flicker↓ | PSNR↑ |", "|---|---|---|"]
    for name, r in rows_k.items(): lines.append(f"| {name} | {r['flicker']:.2f} | {r['psnr']:.2f} |")
b = results["bicubic ×4"]; o1 = results["ours 1-step"]; v1 = video_results["1-step"]
lines += ["", "## Headline",
          f"VAE-free pixel DiT, one forward pass per frame: **{o1[0]-b[0]:+.2f} dB** PSNR over bicubic, LPIPS **{b[2]/o1[2]:.1f}× lower**, "
          f"**{o1[3]:.1f} ms/frame** on this GPU; video flicker **{v1['ours: shared noise']['flicker']:.2f}** vs {v1['per-frame (indep. noise)']['flicker']:.2f} for per-frame diffusion "
          f"(bicubic floor {v1['bicubic']['flicker']:.2f}).",
          "", "## Files", f"`{SAVE_DIR}`: pixelstream.pt, results.json, video_results.json, qualitative.png, preview_*.png, demo_1step.mp4/.gif, demo_4step.mp4/.gif, summary.md"]
open(f"{SAVE_DIR}/summary.md", "w").write("\n".join(lines))
from IPython.display import Markdown
display(Markdown("\n".join(lines)))

In [ ]:
#@title 13 · Long-clip drift analysis (96 frames) — why recurrence is an ablation, not the method
@torch.no_grad()
def stream_sr_anchored(lr_frames, steps=1, alpha=0.5, seed=0):
    g = torch.Generator(device=device).manual_seed(seed)
    noise = torch.randn(1, 3, IMG, IMG, device=device, generator=g)
    z_prev, z_mask = torch.zeros(1, 3, IMG, IMG, device=device), torch.zeros(1, 1, IMG, IMG, device=device)
    prev, mask, outs = z_prev, z_mask, []
    for lr in lr_frames:
        lr_up = upsample(lr[None])
        anchor = sample(ema, build_cond(lr_up, z_prev, z_mask), 1, noise.clone())   # no memory, cannot drift
        cond = build_cond(lr_up, prev, mask)
        out = sample(ema, cond, steps, noise.clone())
        mem = out if steps == 1 else sample(ema, cond, 1, noise.clone())           # deterministic memory
        outs.append(out)
        prev, mask = alpha * mem + (1 - alpha) * anchor, torch.ones_like(mask)
    return torch.cat(outs)

TL = 96
tests = {"1-step shared only":      lambda lr: stream_sr(lr, 1, recurrent=False, shared_noise=True),
         "1-step recurrent":        lambda lr: stream_sr(lr, 1, recurrent=True,  shared_noise=True),
         "1-step anchored a=0.5":   lambda lr: stream_sr_anchored(lr, 1, 0.5),
         "1-step anchored a=0.7":   lambda lr: stream_sr_anchored(lr, 1, 0.7),
         "4-step shared only":      lambda lr: stream_sr(lr, 4, recurrent=False, shared_noise=True),
         "4-step anchored a=0.5":   lambda lr: stream_sr_anchored(lr, 4, 0.5)}
print(f"{'variant':<24}{'flicker↓':>9}{'PSNR↑':>8}{'first8→last8':>16}   ({TL} frames, 3 clips)")
for name, fn in tests.items():
    fl, ps, dr = [], [], []
    for vi, face in enumerate(VIDEO_FACES):
        th = motion_thetas(TL, seed=vi); gt = make_sequence(to_float(val_u8[face:face+1]), th)
        o = fn(F.interpolate(gt, size=(LR_SIZE, LR_SIZE), mode="area")); p = psnr(o, gt)
        fl.append(flicker(o, th)); ps.append(p.mean().item()); dr.append(p[-8:].mean().item() - p[:8].mean().item())
    print(f"{name:<24}{np.mean(fl):9.2f}{np.mean(ps):8.2f}{np.mean(dr):+16.2f}")

In [ ]:
#@title 14 · Latent-DiT baseline — same transformer on SD-VAE latents (fair pixel-vs-latent comparison, ~2 h on a T4)
pip("diffusers>=0.30.0")
from diffusers import AutoencoderKL
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval().requires_grad_(False)
VAE_SF = vae.config.scaling_factor
LAT = IMG // 8                                   # 128 px -> 16×16 latent grid (256 tokens, like the pixel model)

@torch.no_grad()
def encode(x):                                   # pixels [-1,1] -> (B,4,16,16), half precision for speed
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        z = vae.encode(x).latent_dist.mean
    return z.float() * VAE_SF
@torch.no_grad()
def decode(z):                                   # (B,4,16,16) -> pixels
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        x = vae.decode(z.float() / VAE_SF).sample
    return x.float().clamp(-1, 1)

with torch.no_grad():                            # scale the latent residual to the same spread as the pixel target
    _hr = to_float(train_u8[:64]); _up = upsample(make_lr(_hr, False))
    RES_SCALE_LAT = ((_hr - _up) * RES_SCALE).std().item() / (encode(_hr) - encode(_up)).std().item()
print(f"RES_SCALE_LAT = {RES_SCALE_LAT:.2f}")

t0 = time.time()                                 # encode every HR training image once (both flip orientations)
z_hr_all = torch.empty(2, N_TRAIN, 4, LAT, LAT, dtype=torch.float16)
for i in range(0, N_TRAIN, 256):
    hr = to_float(train_u8[i:i+256])
    z_hr_all[0, i:i+256] = encode(hr).half().cpu(); z_hr_all[1, i:i+256] = encode(hr.flip(3)).half().cpu()
print(f"HR latents precomputed in {time.time()-t0:.0f}s")

lat_model = PixelDiT(img=LAT, patch=1, in_ch=8, out_ch=4, dim=DIM, depth=DEPTH, heads=HEADS, repa_layer=REPA_LAYER).to(device)
lat_ema = copy.deepcopy(lat_model).eval().requires_grad_(False)
lat_opt = torch.optim.AdamW(lat_model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.02)
lat_scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and AMP_DTYPE == torch.float16))
print(f"latent DiT: {sum(p.numel() for p in lat_model.parameters())/1e6:.1f}M params, {LAT*LAT} tokens")
lat_ckpt, lat_step0 = f"{SAVE_DIR}/latent_dit.pt", 0
if os.path.exists(lat_ckpt):
    ck = torch.load(lat_ckpt, map_location=device)
    lat_model.load_state_dict(ck["model"]); lat_ema.load_state_dict(ck["ema"]); lat_opt.load_state_dict(ck["opt"]); lat_step0 = ck["step"]
    print("resumed latent DiT from step", lat_step0)

@torch.no_grad()
def sample_lat(net, z_up, steps=1, noise=None):
    B = z_up.shape[0]
    x = torch.randn(B, 4, LAT, LAT, device=device) if noise is None else noise
    ts = torch.linspace(1.0, 0.0, steps + 1, device=device)
    for i in range(steps):
        t, t_next = ts[i], ts[i + 1]
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            x0_hat = net(x, t.expand(B), z_up).float()
        x = x0_hat + (t_next / t) * (x - x0_hat)
    return decode(z_up + x / RES_SCALE_LAT)

lat_model.train(); t_start = time.time()
for step in range(lat_step0, STEPS):
    for g in lat_opt.param_groups: g["lr"] = lr_at(step)
    idx = torch.randint(0, N_TRAIN, (BATCH,)); flip = int(random.random() < 0.5)
    hr = to_float(train_u8[idx]); hr = hr.flip(3) if flip else hr
    z_hr = z_hr_all[flip, idx].to(device).float()
    z_up = encode(upsample(make_lr(hr, DEGRADE)))
    x0 = (z_hr - z_up) * RES_SCALE_LAT
    t = sample_t(BATCH); tb = t.view(-1, 1, 1, 1)
    x_t = (1 - tb) * x0 + tb * torch.randn_like(x0)
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        pred, z = lat_model(x_t, t, z_up, return_repa=True)
    loss_x = F.mse_loss(pred.float(), x0)
    loss_repa = (1 - F.cosine_similarity(z.float(), dino_tokens(hr), dim=-1)).mean() if USE_REPA else torch.zeros((), device=device)
    loss = loss_x + LAMBDA_REPA * loss_repa
    lat_opt.zero_grad(set_to_none=True); lat_scaler.scale(loss).backward()
    lat_scaler.unscale_(lat_opt); torch.nn.utils.clip_grad_norm_(lat_model.parameters(), 1.0)
    lat_scaler.step(lat_opt); lat_scaler.update()
    decay = min(0.9995, (1 + step) / (10 + step))
    with torch.no_grad():
        for pe, pm in zip(lat_ema.parameters(), lat_model.parameters()): pe.lerp_(pm, 1 - decay)
    if (step + 1) % 200 == 0:
        rate = (step + 1 - lat_step0) / (time.time() - t_start)
        print(f"latent step {step+1}/{STEPS} | loss {loss.item():.4f} (x {loss_x.item():.4f}) | {rate:.2f} it/s | ETA {(STEPS-step-1)/rate/60:.0f} min")
    if (step + 1) % 1000 == 0 or step + 1 == STEPS:
        torch.save({"model": lat_model.state_dict(), "ema": lat_ema.state_dict(), "opt": lat_opt.state_dict(), "step": step + 1}, lat_ckpt)

# ---- evaluation with the SAME metrics and the same held-out faces ----
val_zup = torch.cat([encode(val_up[i:i+64]) for i in range(0, N_VAL, 64)])
def run_lat(steps, seed=0):
    g = torch.Generator(device=device).manual_seed(seed); outs = []
    for i in range(0, N_VAL, 64):
        c = val_zup[i:i+64]; outs.append(sample_lat(lat_ema, c, steps, torch.randn(c.shape[0], 4, LAT, LAT, device=device, generator=g)))
    return torch.cat(outs)
def lat_latency_ms(steps, n=50):            # includes VAE encode of the input and decode of the output
    x = val_up[:1]
    for _ in range(5): sample_lat(lat_ema, encode(x), steps)
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n): sample_lat(lat_ema, encode(x), steps)
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n * 1000

cmp = {"bicubic ×4": results["bicubic ×4"],
       "VAE reconstruction ceiling": (*metrics(torch.cat([decode(encode(val[i:i+64])) for i in range(0, N_VAL, 64)]), val), float("nan"))}
lat_out = {}
for k in (1, 2, 4, 8):
    cmp[f"pixel DiT {k}-step (ours)"] = results[f"ours {k}-step"]
    lat_out[k] = run_lat(k); cmp[f"latent DiT {k}-step"] = (*metrics(lat_out[k], val), lat_latency_ms(k))
print(f"\n{'method':<28}{'PSNR↑':>8}{'SSIM↑':>8}{'LPIPS↓':>8}{'ms/frame':>10}")
for name, (p, s, l, ms) in cmp.items(): print(f"{name:<28}{p:8.2f}{s:8.3f}{l:8.3f}{ms:10.1f}")
json.dump(cmp, open(f"{SAVE_DIR}/pixel_vs_latent.json", "w"), indent=1)

idx = list(range(6))
grid = np.concatenate([np.concatenate(list(to_img(c)), 0) for c in (val_up[idx], outs_by_steps[1][idx], lat_out[1][idx], val[idx])], 1)
plt.figure(figsize=(10, 14)); plt.imshow(grid); plt.axis("off"); plt.title("bicubic | pixel DiT 1-step | latent DiT 1-step | ground truth"); plt.show()
Image.fromarray(grid).save(f"{SAVE_DIR}/pixel_vs_latent.png")